<a href="https://colab.research.google.com/github/obeabi/ProjectPortfolio/blob/master/SectorLeadershipModern.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install --upgrade yfinance

In [24]:
import seaborn as sns
import yfinance as yf
print(yf.__version__)
import pandas as pd
#import pandas_ta as ta
import numpy as np
from datetime import datetime
import time
import matplotlib.pyplot as plt


print("Libraries Installed!")

1.4.1
Libraries Installed!


## Define Sector and Benchmark

In [48]:
tickers = [
    'XLK','XLF','XLV','XLI','XLY', 'XBI', 'GBTC',
    'XLP','XLE','XLU','XLB','XLRE','SOXX', 'GBTC', 'GLD','SLV',
    'XLC','SPY'
]

prices = yf.download(
    tickers,
    start="2020-01-01",
    auto_adjust=True
)['Close']

weekly = prices.resample('W-FRI').last()

current_week_end = weekly.index[-1]
previous_week_end = weekly.index[-2]

# =============================
# 2. SCORE FUNCTION (FIXED RS + ACCEL)
# =============================
def calculate_score(data, as_of):

    df = data.loc[:as_of]

    sector = df.drop(columns="SPY")

    # -----------------------------
    # Relative Strength
    # -----------------------------
    rs = sector.div(df["SPY"], axis=0)

    # -----------------------------
    # RS Momentum (4-week change)
    # -----------------------------
    rs_momentum = rs.pct_change(4)

    # -----------------------------
    # RS Acceleration (change in momentum)
    # -----------------------------
    rs_accel = rs_momentum.diff(1)

    # -----------------------------
    # RS Slope (trend of RS)
    # -----------------------------
    rs_slope = (
        rs.rolling(4).mean().iloc[-1] -
        rs.rolling(4).mean().iloc[-4]
    )

    # -----------------------------
    # Latest snapshot
    # -----------------------------
    score = pd.DataFrame({
        "RS_Strength": rs.iloc[-1],
        "RS_Momentum": rs_momentum.iloc[-1],
        "RS_Accel": rs_accel.iloc[-1],
        "RS_Slope": rs_slope
    })

    # Composite rank score
    score["Total"] = score.rank(ascending=False).mean(axis=1)

    return score


# =============================
# 3. CURRENT / PREVIOUS SCORES
# =============================
current_score = calculate_score(weekly, current_week_end)
previous_score = calculate_score(weekly, previous_week_end)

# =============================
# 4. RANKS
# =============================
curr_rank = current_score["Total"].rank()
prev_rank = previous_score["Total"].rank()

# =============================
# 5. IMPACT (NOW USING ACCELERATION)
# =============================
impact = (prev_rank - curr_rank) * current_score["RS_Accel"].abs()

# =============================
# 6. ROTATION TABLE
# =============================
df = pd.DataFrame({
    "Curr Rank": curr_rank,
    "Prev Rank": prev_rank,
    "Rank Change": prev_rank - curr_rank,
    "RS_Strength": current_score["RS_Strength"],
    "RS_Momentum": current_score["RS_Momentum"],
    "RS_Accel": current_score["RS_Accel"],
    "Impact": impact
})

df = df.sort_values("Impact", ascending=False)

# =============================
# 7. TOP 5 ROTATION RATE
# =============================
N = 5

top_now = set(current_score.sort_values("Total").head(N).index)
top_prev = set(previous_score.sort_values("Total").head(N).index)

rotating_in = top_now - top_prev
rotating_out = top_prev - top_now

rotation_rate = len(rotating_in) / N

# =============================
# 8. REGIME CLASSIFICATION
# =============================
if rotation_rate == 0:
    regime = "Stable Leadership"
elif rotation_rate <= 0.2:
    regime = "Low Rotation"
elif rotation_rate <= 0.4:
    regime = "Mild Rotation"
elif rotation_rate <= 0.6:
    regime = "Moderate Rotation"
else:
    regime = "High Rotation"

# =============================
# 9. FLOW LABELS
# =============================
def flow(x):
    if x >= 3:
        return "Strong Inflow 🚀"
    elif x >= 1:
        return "Mild Inflow 📈"
    elif x <= -3:
        return "Strong Outflow 🔻"
    elif x <= -1:
        return "Mild Outflow 📉"
    else:
        return "Neutral"

df["Flow"] = df["Rank Change"].apply(flow)

# =============================
# 10. OUTPUT
# =============================
print("\n==============================")
print("SECTOR ROTATION ENGINE (v2)")
print("==============================")

print(f"Current Week  : {current_week_end.date()}")
print(f"Previous Week : {previous_week_end.date()}")
print(f"Rotation Rate : {rotation_rate:.0%}")
print(f"Regime        : {regime}")

print("\nRotating IN:")
print(rotating_in if rotating_in else "None")

print("\nRotating OUT:")
print(rotating_out if rotating_out else "None")

print("\n==============================")
print("IMPACT TABLE")
print("==============================")

df

[*********************100%***********************]  17 of 17 completed


SECTOR ROTATION ENGINE (v2)
Current Week  : 2026-06-26
Previous Week : 2026-06-19
Rotation Rate : 20%
Regime        : Low Rotation

Rotating IN:
{'XLF'}

Rotating OUT:
{'XLK'}

IMPACT TABLE


,Curr Rank,Prev Rank,Rank Change,RS_Strength,RS_Momentum,RS_Accel,Impact,Flow
Ticker,,,,,,,,
XLE,7.5,16.0,8.5,0.073893,-0.002360,0.097441,0.828245,Strong Inflow 🚀
XLU,7.5,12.0,4.5,0.062528,0.068910,0.085906,0.386578,Strong Inflow 🚀
XLP,6.0,11.0,5.0,0.114447,0.048775,0.070431,0.352156,Strong Inflow 🚀
XLRE,10.0,14.5,4.5,0.060661,0.049605,0.069290,0.311805,Strong Inflow 🚀
XLV,2.0,5.0,3.0,0.212597,0.077939,0.085234,0.255701,Strong Inflow 🚀
XBI,1.0,3.0,2.0,0.206117,0.138895,0.074398,0.148796,Mild Inflow 📈
XLF,5.0,6.0,1.0,0.073076,0.072727,0.045510,0.045510,Mild Inflow 📈
XLI,3.5,4.0,0.5,0.249976,0.092128,0.043171,0.021585,Neutral
XLC,13.5,10.0,-3.5,0.144298,-0.056435,-0.000553,-0.001936,Strong Outflow 🔻


In [49]:
# =============================
# SECTOR SIGNAL CLASSIFICATION
# =============================

def sector_signal(row, rank_series):

    rank = rank_series[row.name]
    accel = row["RS_Accel"]
    rank_change = row["Rank Change"]

    # Top, mid, bottom segmentation
    if rank <= 5:
        tier = "top"
    elif rank <= 10:
        tier = "mid"
    else:
        tier = "weak"

    # -----------------------------
    # BUY CONDITIONS
    # -----------------------------
    if tier == "top" and accel > 0 and rank_change > 0:
        return "BUY 🟢"

    if tier == "top" and accel > 0:
        return "BUY 🟢 (early)"

    # -----------------------------
    # WATCH CONDITIONS
    # -----------------------------
    if tier == "mid" and accel >= 0:
        return "WATCH 🟡 (improving)"

    if tier == "top" and accel <= 0:
        return "WATCH 🟡 (late cycle)"

    if tier == "mid" and rank_change > 0:
        return "WATCH 🟡 (building)"

    # -----------------------------
    # AVOID CONDITIONS
    # -----------------------------
    if tier == "weak" and accel < 0:
        return "AVOID 🔴"

    if rank_change < 0 and accel < 0:
        return "AVOID 🔴 (distribution)"

    return "WATCH 🟡"

In [50]:
df["Signal"] = df.apply(
    sector_signal,
    axis=1,
    rank_series=curr_rank
)

final_view = df.sort_values(
    ["Signal", "Impact"],
    ascending=[True, False]
)

print("\n==============================")
print("SECTOR SIGNAL DASHBOARD")
print("==============================")

final_view


SECTOR SIGNAL DASHBOARD


,Curr Rank,Prev Rank,Rank Change,RS_Strength,RS_Momentum,RS_Accel,Impact,Flow,Signal
Ticker,,,,,,,,,
XLC,13.5,10.0,-3.5,0.144298,-0.056435,-0.000553,-0.001936,Strong Outflow 🔻,AVOID 🔴
XLY,12.0,9.0,-3.0,0.154688,-0.032427,-0.011507,-0.034522,Strong Outflow 🔻,AVOID 🔴
GLD,13.5,8.0,-5.5,0.504470,-0.087455,-0.019156,-0.105359,Strong Outflow 🔻,AVOID 🔴
SLV,16.0,14.5,-1.5,0.071686,-0.208410,-0.075433,-0.113149,Mild Outflow 📉,AVOID 🔴
XLK,9.0,2.0,-7.0,0.251475,-0.005479,-0.062449,-0.437146,Strong Outflow 🔻,AVOID 🔴 (distribution)
XLV,2.0,5.0,3.0,0.212597,0.077939,0.085234,0.255701,Strong Inflow 🚀,BUY 🟢
XBI,1.0,3.0,2.0,0.206117,0.138895,0.074398,0.148796,Mild Inflow 📈,BUY 🟢
XLF,5.0,6.0,1.0,0.073076,0.072727,0.045510,0.045510,Mild Inflow 📈,BUY 🟢
XLI,3.5,4.0,0.5,0.249976,0.092128,0.043171,0.021585,Neutral,BUY 🟢
